# Notebook 01: Extração e Pré-Processamento de Dados

Este notebook é responsável por acessar o banco de dados mestre da PNS 2019 (`pns_master_formatado.db`) e criar quatro sub-bancos focados em pessoas idosas (≥60 anos) conforme os critérios da pesquisa:

1. `idosos_geral.db` (Todos os idosos)
2. `idosos_saudaveis.db` (Idosos sem NENHUMA doença crônica reportada)
3. `idosos_artrite.db` (Idosos com diagnóstico de artrite/reumatismo)
4. `idosos_artrite_puro.db` (Idosos com artrite, mas SEM as outras comorbidades)

In [ ]:
import sqlite3
import pandas as pd
import os
import gc

# Configuração de diretórios
PASTA_DB = '../data/database/'
DB_MASTER = os.path.join(PASTA_DB, 'pns_master_formatado.db')

print(f"Verificando existência do banco mestre: {os.path.exists(DB_MASTER)}")

## 1. Definição das Variáveis de Doenças Crônicas
Vamos utilizar as 14 variáveis do **Módulo Q** para classificar os perfis.

In [ ]:
# Q079 = Artrite ou reumatismo
variaveis_doencas = [
    'Q00201', # Hipertensão
    'Q03001', # Diabetes
    'Q060',   # Colesterol
    'Q06306', # Doença do coração
    'Q068',   # AVC
    'Q074',   # Asma
    'Q079',   # Artrite/Reumatismo (Alvo)
    'Q088',   # DORT
    'Q092',   # Depressão
    'Q11006', # Outra doença mental
    'Q11604', # Doença crônica no pulmão
    'Q120',   # Câncer
    'Q124',   # Insuficiência renal crônica
    'Q128'    # Outra doença crônica
]

# Vamos criar uma string SQL para selecionar idosos (C008 >= 60)
# A coluna C008 (idade) está armazenada como texto no SQLite, então usamos CAST
filtro_idosos = "CAST(C008 AS INTEGER) >= 60"

## 2. Extração dos Dados (Idosos Geral)
Extraímos todos os idosos do banco e salvamos no primeiro banco de dados.

In [ ]:
print("Extraindo Idosos Geral...")
conn_master = sqlite3.connect(DB_MASTER)

# Ler todos os idosos do banco mestre
query_geral = f"SELECT * FROM pns_completa WHERE {filtro_idosos}"
df_idosos = pd.read_sql_query(query_geral, conn_master)
conn_master.close()

print(f"Total de Idosos encontrados: {len(df_idosos)}")

# Salvar em banco separado
conn_geral = sqlite3.connect(os.path.join(PASTA_DB, 'idosos_geral.db'))
df_idosos.to_sql('pns_idosos', conn_geral, if_exists='replace', index=False)
conn_geral.close()
print("Banco 'idosos_geral.db' salvo com sucesso.")

## 3. Criação dos Subgrupos (Saudáveis, Artrite e Artrite Pura)

In [ ]:
# A string no banco está com acento (Não). Como pode haver problema de encoding, 
# vamos tratar verificando se começa com "N" ou "S".
def eh_nao(val):
    if pd.isna(val): return False
    return str(val).strip().upper().startswith('N')

def eh_sim(val):
    if pd.isna(val): return False
    return str(val).strip().upper().startswith('S')

# Máscaras booleanas
# 1. Tem artrite (Q079 = Sim)
mask_artrite = df_idosos['Q079'].apply(eh_sim)

# 2. Tem outras doenças?
# Avaliamos as outras 13 doenças
doencas_sem_artrite = [v for v in variaveis_doencas if v != 'Q079']

# map foi depreciado para DataFrames inteiros, melhor usar applymap para pandas mais antigos, mas como temos pandas recente:
try:
    mask_outras_doencas = df_idosos[doencas_sem_artrite].map(eh_sim).any(axis=1)
except AttributeError:
    mask_outras_doencas = df_idosos[doencas_sem_artrite].applymap(eh_sim).any(axis=1)

# 3. É saudável? (Q079 não e NENHUMA outra sim) -> Precisamos confirmar que todas as 14 são 'Não'
try:
    mask_saudavel = df_idosos[variaveis_doencas].map(eh_nao).all(axis=1)
except AttributeError:
    mask_saudavel = df_idosos[variaveis_doencas].applymap(eh_nao).all(axis=1)

# Aplicando as máscaras
df_artrite = df_idosos[mask_artrite]
df_artrite_puro = df_idosos[mask_artrite & ~mask_outras_doencas]
df_saudaveis = df_idosos[mask_saudavel]

print(f"Idosos com Artrite: {len(df_artrite)}")
print(f"Idosos com Artrite Pura: {len(df_artrite_puro)}")
print(f"Idosos Saudáveis: {len(df_saudaveis)}")

## 4. Salvando os Bancos de Dados Resultantes

In [ ]:
# Salvando Artrite
conn_artrite = sqlite3.connect(os.path.join(PASTA_DB, 'idosos_artrite.db'))
df_artrite.to_sql('pns_idosos', conn_artrite, if_exists='replace', index=False)
conn_artrite.close()

# Salvando Artrite Pura
conn_artrite_puro = sqlite3.connect(os.path.join(PASTA_DB, 'idosos_artrite_puro.db'))
df_artrite_puro.to_sql('pns_idosos', conn_artrite_puro, if_exists='replace', index=False)
conn_artrite_puro.close()

# Salvando Saudáveis
conn_saudaveis = sqlite3.connect(os.path.join(PASTA_DB, 'idosos_saudaveis.db'))
df_saudaveis.to_sql('pns_idosos', conn_saudaveis, if_exists='replace', index=False)
conn_saudaveis.close()

print("Todos os bancos foram recriados e salvos com sucesso!")

## 5. Próximo Passo: Preparação para Análises

Com os bancos de idosos criados corretamente, você já pode seguir para o **Notebook 02** para as análises estatísticas ou cruzar variáveis (obesidade, exercícios) de dentro dessas bases.